# azroute: training run

Improves all 45 challenge cases while training the network on the search's results, then exports
checked submission files. Built for a **G4** GPU runtime (Runtime > Change runtime type).

Everything is checkpointed to Google Drive every 30 minutes. After a disconnect, run sections 1-3
again: the run resumes from Drive and keeps its original deadline. To start over, use a new `DRIVE_DIR`.

## 1. Settings

In [ ]:
#@title Settings
REPO_URL = "https://github.com/Dev077/azroute.git"  #@param {type:"string"}
DRIVE_DIR = "/content/drive/MyDrive/azroute_run"  #@param {type:"string"}
HOURS = 23.5  #@param {type:"number"}
#@markdown `HOURS`: total wall time, including the final export. For a private repo, put a token in
#@markdown the URL: `https://<token>@github.com/you/azroute.git`.

CHALLENGE_URL = "https://github.com/partcleda/eda-3d-routing-challenge.git"
CHALLENGE_DIR = "/content/eda-3d-routing-challenge"
CODE_DIR = "/content/azroute"
WORK_DIR = "/content/work"

## 2. Setup
Mounts Drive, fetches the challenge toolkit and this repository, builds the C++ core and runs the tests.

In [ ]:
#@title Setup
import os, subprocess
from google.colab import drive

drive.mount('/content/drive')

def sh(cmd, cwd='/content'):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode:
        raise RuntimeError(f'failed: {cmd}')

for url, path in [(CHALLENGE_URL, CHALLENGE_DIR), (REPO_URL, CODE_DIR)]:
    if os.path.isdir(path):
        sh(f'git -C {path} pull --ff-only')
    else:
        sh(f'git clone --depth 1 {url} {path}')
os.environ['AZROUTE_CHALLENGE_ROOT'] = CHALLENGE_DIR
os.environ['PYTHONPATH'] = CODE_DIR
sh('pip -q install pybind11 psutil')
sh('python setup.py > build.log 2>&1 || (tail -40 build.log; exit 1)', cwd=CODE_DIR)
sh('python -m unittest tests.test_core', cwd=CODE_DIR)

import torch, psutil
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > G4'
print(torch.cuda.get_device_name(0), '|', os.cpu_count(), 'CPUs |',
      round(psutil.virtual_memory().total / 2**30), 'GB RAM')

## 3. Run
Starts the run in the background, or resumes it from Drive. Safe to re-run: it does nothing if the run is already going.

In [ ]:
#@title Start or resume
# '[p]ipeline[.]run' matches the run's command line but not this check's own shell
running = subprocess.run("pgrep -f '[p]ipeline[.]run'", shell=True, capture_output=True, text=True).stdout.split()
if running:
    print('already running, pid', running)
else:
    os.makedirs(WORK_DIR, exist_ok=True)
    args = ['python', '-m', 'pipeline.run', '--work', WORK_DIR, '--drive', DRIVE_DIR]
    if not os.path.exists(os.path.join(DRIVE_DIR, 'run_config.json')):
        args += ['--hours', str(HOURS)]          # fresh run; on resume the stored deadline is kept
    log = open(os.path.join(WORK_DIR, 'run.log'), 'a')
    p = subprocess.Popen(args, cwd=CODE_DIR, env=dict(os.environ), stdout=log, stderr=subprocess.STDOUT,
                         start_new_session=True)
    print('started pid', p.pid, ' '.join(args))

## 4. Monitor
Re-run any time: the latest log lines, the score per tier, and three charts (training losses, score over time, network vs network-free reroutes).

In [ ]:
#@title Monitor
import json, math, collections
import matplotlib.pyplot as plt

def jsonl(name):
    path = os.path.join(WORK_DIR, name)
    return [json.loads(l) for l in open(path)] if os.path.exists(path) else []

def geomean(pairs):
    return math.exp(sum(math.log(b / d) for b, d in pairs) / len(pairs))

log_path = os.path.join(WORK_DIR, 'run.log')
print(''.join(open(log_path).readlines()[-15:]) if os.path.exists(log_path) else 'no run.log yet: start the run in section 3')

stats_path = os.path.join(WORK_DIR, 'case_stats.json')
stats = {k: v for k, v in json.load(open(stats_path)).items() if not k.startswith('_')} \
    if os.path.exists(stats_path) else {}
by_tier = collections.defaultdict(list)
for s in stats.values():
    if s.get('best'):
        by_tier[s['tier']].append(s)
print(f"{'tier':10} {'cases':>5} {'vs baseline':>12} {'vs initial':>11} {'delay/LB':>9}")
for t, rs in by_tier.items():
    vs_base = geomean([(s['baseline'], s['best']) for s in rs])
    vs_init = geomean([(s.get('initial_delay', s['best']), s['best']) for s in rs])
    vs_lb = sum(s['best'] for s in rs) / max(1, sum(s.get('lower_bound', 0) for s in rs))
    print(f"{t:10} {len(rs):5d} {vs_base:12.4f} {vs_init:11.4f} {vs_lb:9.4f}")

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
metrics = jsonl('metrics.jsonl')
for key in ['policy_kl', 'value_mse', 'future_bce', 'blocked_bce']:
    ax[0].plot([r['step'] for r in metrics], [r[key] for r in metrics], marker='.', label=key)
ax[0].set_title('training losses'); ax[0].set_xlabel('step'); ax[0].set_yscale('log')
if metrics:
    ax[0].legend()

visits = sorted(jsonl('visits.jsonl'), key=lambda r: r['t'])
# every case counts from its initial routing, so the line is the tier score throughout
best = {k: s['initial_delay'] for k, s in stats.items() if s.get('initial_delay')}
series = collections.defaultdict(list)
for r in visits:
    if 'delay' not in r or r['case'] not in stats:
        continue
    best[r['case']] = r['delay']
    tier = r['case'].split('/')[0]
    pairs = [(stats[k]['baseline'], d) for k, d in best.items() if k.startswith(tier + '/')]
    series[tier].append(((r['t'] - visits[0]['t']) / 3600, geomean(pairs)))
for tier, pts in series.items():
    ax[1].plot([p[0] for p in pts], [p[1] for p in pts], label=tier)
ax[1].set_title('score vs baseline'); ax[1].set_xlabel('hours')
if series:
    ax[1].legend()

# only since the first network visit, so both bars cover the same hours
searched = [r for r in visits if r.get('reroutes')]
t_net = min((r['t'] for r in searched if r.get('network')), default=None)
for flag, label in [(False, 'network-free'), (True, 'network')]:
    rr = [r for r in searched if r.get('network') == flag and t_net is not None and r['t'] >= t_net]
    if rr:
        n = sum(r['reroutes'] for r in rr)
        ax[2].bar(label, sum(r['better'] for r in rr) / max(1, n))
        ax[2].text(label, 0, f' n={n}', rotation=90, va='bottom')
ax[2].set_title('share of reroutes that improve')
plt.tight_layout(); plt.show()

## 5. Stop early (optional)
Workers finish their current reroute; the run then syncs to Drive and exports, as it would at the deadline.

In [ ]:
#@title Stop
open(os.path.join(WORK_DIR, 'STOP'), 'w').close()
print('stop requested; watch the log in section 4')
# To kill the run outright (no export), kill its whole process group; the worker processes'
# command lines do not contain "pipeline.run", so pkill on the name alone leaves them running:
#   !kill -TERM -- -$(pgrep -f '[p]ipeline[.]run' | head -1)

## 6. Export
The run exports automatically when it ends. This writes the checked submission files from the current routings at any time, to `DRIVE_DIR/export/`, and zips them.

In [ ]:
#@title Export now
out = os.path.join(DRIVE_DIR, 'export')
sh(f'python -m pipeline.export --work {WORK_DIR} --out {out}', cwd=CODE_DIR)
sh(f'cd {out} && zip -qr submissions.zip submissions summary.md && ls -la')